# UPI App Decline Analysis
## Notebook 01 — Data Cleaning (Bronze to Silver to Gold)
---
**What this notebook does (and ONLY this):**
- Loads all raw CSV files **(Bronze)**
- Cleans and standardises each file **(Silver)**
- Merges into one master table **(Gold)**
- Creates the master gold table for **(SQL Server)**

**What this notebook does NOT do:** No SQL analysis. No charts. No visualisations.

---
**Files needed in the same folder as this notebook:**
```
gpay_help.csv
gpay_new.csv
paytm_help.csv
paytm_new.csv
phonepe_help.csv
phonepe_new.csv
playstore_reviews.csv
news_articles.csv
```

## Step 1 — Imports

In [1]:
import pandas as pd
import os

pd.set_option("display.max_colwidth", 80)
print("Imports done")
print(f"  pandas version : {pd.__version__}")


Imports done
  pandas version : 2.2.3


## Step 2 — File Check (Bronze Layer)
Before loading anything, verify all required files are present.


In [2]:
EXPECTED_FILES = [
    'gpay_help.csv',
    'gpay_new.csv',
    'paytm_help.csv',
    'paytm_new.csv',
    'phonepe_help.csv',
    'phonepe_new.csv',
    'playstore_reviews.csv',
    'news_articles.csv',
]

print('=== FILE CHECK ===')
all_present = True
for f in EXPECTED_FILES:
    exists = os.path.exists(f)
    status = 'FOUND  ' if exists else 'MISSING'
    size   = f'{os.path.getsize(f)/1024/1024:.1f} MB' if exists else ''
    print(f'  [{status}]  {f:45s} {size}')
    if not exists:
        all_present = False

print()
if all_present:
    print('All files present. Ready to proceed.')
else:
    print('WARNING: Some files missing. Add them before continuing.')


=== FILE CHECK ===
  [FOUND  ]  gpay_help.csv                                 25.8 MB
  [FOUND  ]  gpay_new.csv                                  11.3 MB
  [FOUND  ]  paytm_help.csv                                22.7 MB
  [FOUND  ]  paytm_new.csv                                 15.2 MB
  [FOUND  ]  phonepe_help.csv                              21.5 MB
  [FOUND  ]  phonepe_new.csv                               10.1 MB
  [FOUND  ]  playstore_reviews.csv                         0.9 MB
  [FOUND  ]  news_articles.csv                             0.1 MB

All files present. Ready to proceed.


## Step 3 — Load Kaggle Files (Bronze)
Load all 6 Kaggle CSVs exactly as-is. No changes yet.
We only tag each row with app name and data source.


In [3]:
KAGGLE_FILES = {
    'gpay_help.csv':    {'app': 'GPay',    'source': 'kaggle_help'},
    'gpay_new.csv':     {'app': 'GPay',    'source': 'kaggle_new'},
    'paytm_help.csv':   {'app': 'Paytm',   'source': 'kaggle_help'},
    'paytm_new.csv':    {'app': 'Paytm',   'source': 'kaggle_new'},
    'phonepe_help.csv': {'app': 'PhonePe', 'source': 'kaggle_help'},
    'phonepe_new.csv':  {'app': 'PhonePe', 'source': 'kaggle_new'},
}

bronze_frames = []
print('=== LOADING KAGGLE FILES (BRONZE) ===')
print()
for filename, meta in KAGGLE_FILES.items():
    df = pd.read_csv(filename)
    df['app']         = meta['app']
    df['data_source'] = meta['source']
    bronze_frames.append(df)
    print(f'  {filename:25s}  {len(df):>6,} rows')

df_bronze_kaggle = pd.concat(bronze_frames, ignore_index=True)
print()
print(f'Total Bronze (Kaggle): {len(df_bronze_kaggle):,} rows')
print(f'Columns: {list(df_bronze_kaggle.columns)}')


=== LOADING KAGGLE FILES (BRONZE) ===

  gpay_help.csv              40,000 rows
  gpay_new.csv               40,000 rows
  paytm_help.csv             40,000 rows
  paytm_new.csv              40,000 rows
  phonepe_help.csv           40,000 rows
  phonepe_new.csv            40,000 rows

Total Bronze (Kaggle): 240,000 rows
Columns: ['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion', 'app', 'data_source']


## Step 4 — Load Fresh Scrape 2026 (Bronze)
The fresh scrape has different column names — that is expected.
Load as-is. We will align the schema in Silver.


In [4]:
df_bronze_fresh = pd.read_csv('playstore_reviews.csv', encoding='utf-8-sig')
df_bronze_fresh['data_source'] = 'fresh_scrape_2026'

print('=== FRESH SCRAPE (BRONZE) ===')
print(f'  Rows    : {len(df_bronze_fresh):,}')
print(f'  Columns : {list(df_bronze_fresh.columns)}')
print(f'  Apps    : {df_bronze_fresh["app"].unique().tolist()}')
print(f'  Dates   : {df_bronze_fresh["date"].min()} to {df_bronze_fresh["date"].max()}')
print()
print(df_bronze_fresh.head(3).to_string())


=== FRESH SCRAPE (BRONZE) ===
  Rows    : 8,000
  Columns : ['app', 'review_id', 'date', 'year_month', 'rating', 'review_text', 'thumbs_up', 'app_version', 'data_source']
  Apps    : ['BharatPe', 'GPay', 'Paytm', 'PhonePe']
  Dates   : 2026-06-08 to 2026-06-27

        app                             review_id        date year_month  rating  review_text  thumbs_up app_version        data_source
0  BharatPe  77741b23-1899-470e-aacc-8a9d4ca96866  2026-06-26    2026-06       1  bad service          0       7.4.9  fresh_scrape_2026
1  BharatPe  5d16af31-c7ab-4140-bbfd-a7eef4dc81cd  2026-06-26    2026-06       5         naic          0       7.4.2  fresh_scrape_2026
2  BharatPe  741f3317-2106-46b7-8b80-26d8fa4015e8  2026-06-26    2026-06       5    very nice          0       7.4.9  fresh_scrape_2026


## Step 5 — Bronze Layer Snapshot
Record what raw data looks like before any cleaning.
Think of this as your audit log.


In [5]:
print('=' * 55)
print('  BRONZE LAYER SUMMARY')
print('=' * 55)
print()
print('Kaggle files:')
print(df_bronze_kaggle.groupby(['app','data_source'])
      .size().reset_index(name='rows').to_string(index=False))
print()
print('Fresh scrape (2026):')
print(df_bronze_fresh.groupby('app').size()
      .reset_index(name='rows').to_string(index=False))
total = len(df_bronze_kaggle) + len(df_bronze_fresh)
print()
print(f'Total raw rows: {total:,}')
print('=' * 55)


  BRONZE LAYER SUMMARY

Kaggle files:
    app data_source  rows
   GPay kaggle_help 40000
   GPay  kaggle_new 40000
  Paytm kaggle_help 40000
  Paytm  kaggle_new 40000
PhonePe kaggle_help 40000
PhonePe  kaggle_new 40000

Fresh scrape (2026):
     app  rows
BharatPe  2000
    GPay  2000
   Paytm  2000
 PhonePe  2000

Total raw rows: 248,000


## Step 6 — Clean Kaggle Data (Silver Layer)

**Every action has a reason:**

| Action | Why |
|---|---|
| Rename columns | Standardise - `content` to `review_text`, `score` to `rating`, `at` to `date` |
| Drop `userImage` | URL column - no analytical value |
| Parse dates | So SQL can run time-series queries properly |
| Add `year` and `year_month` | Pre-computed for GROUP BY in SQL |
| Drop null review_text | Cannot analyse empty reviews |
| Strip whitespace | Removes invisible characters that break keyword matching |
| Deduplicate | Same reviewId twice = double counting |
| Filter short reviews | Under 5 chars (like 'ok') add no signal |


In [6]:
df_silver_kaggle = df_bronze_kaggle.copy()

# 1. Rename to standard column names
df_silver_kaggle = df_silver_kaggle.rename(columns={
    'reviewId':             'review_id',
    'userName':             'user_name',
    'content':              'review_text',
    'score':                'rating',
    'thumbsUpCount':        'thumbs_up',
    'reviewCreatedVersion': 'app_version',
    'at':                   'date',
    'replyContent':         'reply_text',
    'repliedAt':            'reply_date',
})

# 2. Drop columns with no analytical value
df_silver_kaggle = df_silver_kaggle.drop(columns=['userImage'], errors='ignore')

# 3. Parse dates
df_silver_kaggle['date'] = pd.to_datetime(df_silver_kaggle['date'], errors='coerce')

# 4. Add time columns for SQL GROUP BY
df_silver_kaggle['year']       = df_silver_kaggle['date'].dt.year
df_silver_kaggle['year_month'] = df_silver_kaggle['date'].dt.to_period('M').astype(str)

# 5. Clean review text
df_silver_kaggle['review_text'] = (
    df_silver_kaggle['review_text'].fillna('').astype(str).str.strip()
)

# 6. Drop short/empty reviews
before = len(df_silver_kaggle)
df_silver_kaggle = df_silver_kaggle[df_silver_kaggle['review_text'].str.len() >= 5]
print(f'Dropped {before - len(df_silver_kaggle):,} short/empty reviews')

# 7. Deduplicate
before = len(df_silver_kaggle)
df_silver_kaggle = df_silver_kaggle.drop_duplicates(subset='review_id')
print(f'Dropped {before - len(df_silver_kaggle):,} duplicate reviews')

# 8. Drop rows where date parsing failed
before = len(df_silver_kaggle)
df_silver_kaggle = df_silver_kaggle.dropna(subset=['date'])
print(f'Dropped {before - len(df_silver_kaggle):,} rows with bad dates')

# 9. Keep only valid ratings
df_silver_kaggle = df_silver_kaggle[df_silver_kaggle['rating'].between(1, 5)]

print()
print(f'Kaggle Silver rows : {len(df_silver_kaggle):,}')
print(f'Date range         : {df_silver_kaggle["date"].min().date()} to {df_silver_kaggle["date"].max().date()}')
print(f'Columns            : {list(df_silver_kaggle.columns)}')

Dropped 36,535 short/empty reviews
Dropped 17,956 duplicate reviews
Dropped 0 rows with bad dates

Kaggle Silver rows : 185,509
Date range         : 2018-09-12 to 2023-07-18
Columns            : ['review_id', 'user_name', 'review_text', 'rating', 'thumbs_up', 'app_version', 'date', 'reply_text', 'reply_date', 'appVersion', 'app', 'data_source', 'year', 'year_month']


## Step 7 — Clean Fresh Scrape (Silver Layer)
Align the fresh scrape schema with the Kaggle Silver schema
so both can be merged cleanly in Step 8.


In [7]:
df_silver_fresh = df_bronze_fresh.copy()

# 1. Parse dates
df_silver_fresh['date'] = pd.to_datetime(df_silver_fresh['date'], errors='coerce')

# 2. Add year and year_month
df_silver_fresh['year']       = df_silver_fresh['date'].dt.year
df_silver_fresh['year_month'] = df_silver_fresh['date'].dt.to_period('M').astype(str)

# 3. Add columns Kaggle has but fresh scrape does not
df_silver_fresh['reply_text'] = None
df_silver_fresh['reply_date'] = None
df_silver_fresh['user_name']  = None

# 4. Rename columns to match Kaggle silver schema
df_silver_fresh = df_silver_fresh.rename(columns={
    'review_id':   'review_id',
    'review_text': 'review_text',
    'rating':      'rating',
    'thumbs_up':   'thumbs_up',
    'app_version': 'app_version',
})

# 5. Clean review text
df_silver_fresh['review_text'] = (
    df_silver_fresh['review_text'].fillna('').astype(str).str.strip()
)

# 6. Drop short reviews
before = len(df_silver_fresh)
df_silver_fresh = df_silver_fresh[df_silver_fresh['review_text'].str.len() >= 5]
print(f'Dropped {before - len(df_silver_fresh):,} short reviews')

# 7. Deduplicate
before = len(df_silver_fresh)
df_silver_fresh = df_silver_fresh.drop_duplicates(subset='review_id')
print(f'Dropped {before - len(df_silver_fresh):,} duplicates')

# 8. Keep only Paytm, GPay, PhonePe
df_silver_fresh = df_silver_fresh[
    df_silver_fresh['app'].isin(['Paytm','GPay','PhonePe'])
]
print(f'Filtered to 3 apps: {len(df_silver_fresh):,} rows')

print()
print(f'Fresh Scrape Silver rows : {len(df_silver_fresh):,}')
print(f'Date range               : {df_silver_fresh["date"].min().date()} to {df_silver_fresh["date"].max().date()}')


Dropped 2,559 short reviews
Dropped 0 duplicates
Filtered to 3 apps: 4,138 rows

Fresh Scrape Silver rows : 4,138
Date range               : 2026-06-09 to 2026-06-27


## Step 8 — Build Master Table (Gold Layer)
Merge both Silver tables into one unified master table.
This is the single source of truth for every SQL query.


In [8]:
GOLD_COLUMNS = [
    'review_id', 'app', 'data_source', 'date', 'year',
    'year_month', 'rating', 'review_text', 'thumbs_up',
    'reply_text', 'reply_date', 'user_name', 'app_version'
]

df_gold_kaggle = df_silver_kaggle.reindex(columns=GOLD_COLUMNS)
df_gold_fresh  = df_silver_fresh.reindex(columns=GOLD_COLUMNS)

df_gold = pd.concat([df_gold_kaggle, df_gold_fresh], ignore_index=True)

# Final dedup across both sources
before = len(df_gold)
df_gold = df_gold.drop_duplicates(subset='review_id')
print(f'Cross-source duplicates removed: {before - len(df_gold):,}')

df_gold = df_gold.sort_values(['app','date']).reset_index(drop=True)

print()
print('=' * 55)
print('  GOLD MASTER TABLE')
print('=' * 55)
print(f'  Total rows : {len(df_gold):,}')
print(f'  Columns    : {list(df_gold.columns)}')
print()
print('  Rows by app and source:')
print(df_gold.groupby(['app','data_source']).size()
      .reset_index(name='rows').to_string(index=False))
print()
print('  Year range by app:')
print(df_gold.groupby('app')['year'].agg(['min','max']).to_string())
print()
print('  Avg rating by app:')
print(df_gold.groupby('app')['rating'].mean().round(2).to_string())
print('=' * 55)


Cross-source duplicates removed: 13

  GOLD MASTER TABLE
  Total rows : 189,634
  Columns    : ['review_id', 'app', 'data_source', 'date', 'year', 'year_month', 'rating', 'review_text', 'thumbs_up', 'reply_text', 'reply_date', 'user_name', 'app_version']

  Rows by app and source:
    app       data_source  rows
   GPay fresh_scrape_2026  1488
   GPay       kaggle_help 40000
   GPay        kaggle_new 23066
  Paytm fresh_scrape_2026  1313
  Paytm       kaggle_help 40000
  Paytm        kaggle_new 20878
PhonePe fresh_scrape_2026  1324
PhonePe       kaggle_help 40000
PhonePe        kaggle_new 21565

  Year range by app:
          min   max
app                
GPay     2018  2026
Paytm    2018  2026
PhonePe  2018  2026

  Avg rating by app:
app
GPay       2.59
Paytm      2.90
PhonePe    3.17


## Step 9 — Data Validation
Run checks before writing to the database.
Same concept as the `tests/` folder in your data warehouse project.


In [11]:
print('=== VALIDATION CHECKS ===')
print()
passed = 0
total  = 0

def check(label, condition, detail=''):
    global passed, total
    total += 1
    mark = 'PASS' if condition else 'FAIL'
    if condition: passed += 1
    print(f'  [{mark}]  {label}')
    if detail: print(f'          {detail}')

check('No duplicate review_ids',
      df_gold['review_id'].nunique() == len(df_gold),
      f'{df_gold["review_id"].nunique():,} unique / {len(df_gold):,} total')

check('All ratings 1-5',
      df_gold['rating'].between(1,5).all(),
      f'Min={df_gold["rating"].min()}  Max={df_gold["rating"].max()}')

check('No null dates',
      df_gold['date'].isna().sum() == 0,
      f'Null dates: {df_gold["date"].isna().sum()}')

check('No null review_text',
      df_gold['review_text'].isna().sum() == 0,
      f'Null texts: {df_gold["review_text"].isna().sum()}')

check('Only 3 apps',
      set(df_gold['app'].unique()) == {'Paytm','GPay','PhonePe'},
      f'Apps: {sorted(df_gold["app"].unique().tolist())}')

check('Year range 2018-2026',
      df_gold['year'].between(2018,2026).all(),
      f'Years: {sorted(df_gold["year"].unique().tolist())}')

check(
    'At least 150,000 rows',
    len(df_gold) >= 150_000,
    f'Total: {len(df_gold):,}')

print()
print(f'Result: {passed}/{total} checks passed')
if passed == total:
    print('All checks passed. Safe to write to database.')
else:
    print('Fix failing checks before proceeding.')

=== VALIDATION CHECKS ===

  [PASS]  No duplicate review_ids
          189,634 unique / 189,634 total
  [PASS]  All ratings 1-5
          Min=1  Max=5
  [PASS]  No null dates
          Null dates: 0
  [PASS]  No null review_text
          Null texts: 0
  [PASS]  Only 3 apps
          Apps: ['GPay', 'Paytm', 'PhonePe']
  [PASS]  Year range 2018-2026
          Years: [2018, 2019, 2020, 2021, 2022, 2023, 2026]
  [PASS]  At least 150,000 rows
          Total: 189,634

Result: 7/7 checks passed
All checks passed. Safe to write to database.


## Step 10 — Export Gold Table for SQL Server
This is our final output from the cleaning pipeline.
We export the Gold master table as a CSV which you will import
directly into SQL Server via SSMS.

**Why CSV and not a direct connection?**
BULK INSERT in SQL Server is the fastest and most reliable way
to load large datasets (189,634 rows). It is also the standard
approach used in real ETL pipelines.

| Column | Type in SQL Server | Notes |
|---|---|---|
| review_id | NVARCHAR(100) | Unique identifier |
| app | NVARCHAR(50) | GPay, Paytm, PhonePe |
| data_source | NVARCHAR(50) | kaggle_help / kaggle_new / fresh_scrape_2026 |
| date | DATETIME | Review date |
| year | INT | Extracted from date for GROUP BY |
| year_month | NVARCHAR(10) | e.g. 2023-04 for monthly trends |
| rating | INT | 1 to 5 stars |
| review_text | NVARCHAR(MAX) | Full review content |
| thumbs_up | INT | Helpfulness votes |
| reply_text | NVARCHAR(MAX) | Company reply if any |
| reply_date | DATETIME | When company replied |
| user_name | NVARCHAR(200) | Reviewer name |
| app_version | NVARCHAR(50) | App version at time of review |


In [12]:
import os

EXPORT_PATH = "gold_master_export.csv"

# Convert date columns to string for clean SQL Server import
df_export = df_gold.copy()
df_export["date"] = pd.to_datetime(
    df_export["date"], errors="coerce"
).dt.strftime("%Y-%m-%d %H:%M:%S")
df_export["reply_date"] = pd.to_datetime(
    df_export["reply_date"], errors="coerce"
).dt.strftime("%Y-%m-%d %H:%M:%S")

# Replace NaN with empty string so BULK INSERT handles nulls cleanly
df_export = df_export.fillna("")

# Export
df_export.to_csv(EXPORT_PATH, index=False, encoding="utf-8-sig")

print("=== EXPORT COMPLETE ===")
print(f"  File    : {EXPORT_PATH}")
print(f"  Rows    : {len(df_export):,}")
print(f"  Columns : {list(df_export.columns)}")
print(f"  Size    : {os.path.getsize(EXPORT_PATH)/1024/1024:.1f} MB")
print()
print("Copy this file to a location SQL Server can access.")
print("Then run the SSMS script in Step 11.")

=== EXPORT COMPLETE ===
  File    : gold_master_export.csv
  Rows    : 189,634
  Columns : ['review_id', 'app', 'data_source', 'date', 'year', 'year_month', 'rating', 'review_text', 'thumbs_up', 'reply_text', 'reply_date', 'user_name', 'app_version']
  Size    : 75.7 MB

Copy this file to a location SQL Server can access.
Then run the SSMS script in Step 11.


## Step 11 — Create Database and Import in SSMS

Run the T-SQL script printed below directly in SSMS.
It will:
1. Create the  database
2. Create the  table with correct schema
3. Import  using BULK INSERT
4. Run a verification query to confirm data landed correctly

> **Before running:** Update the file path in the BULK INSERT
> statement to match where you saved .
> Example: 


In [14]:
script = r"""
-- ================================================
-- UPI Analysis -- SSMS Setup Script
-- Run this entire script in SSMS
-- ================================================

-- STEP 1: Create Database

USE master;
GO

IF NOT EXISTS (
    SELECT name 
    FROM sys.databases
    WHERE name = N'UPI_Analysis'
)
BEGIN
    CREATE DATABASE UPI_Analysis;
END
GO


USE UPI_Analysis;
GO


-- STEP 2: Create reviews table

DROP TABLE IF EXISTS dbo.reviews;

CREATE TABLE dbo.reviews (

    review_id    NVARCHAR(100),

    app          NVARCHAR(50),

    data_source  NVARCHAR(50),

    date         DATETIME,

    year         INT,

    year_month   NVARCHAR(10),

    rating       INT,

    review_text  NVARCHAR(MAX),

    thumbs_up    INT,

    reply_text   NVARCHAR(MAX),

    reply_date   DATETIME,

    user_name    NVARCHAR(200),

    app_version  NVARCHAR(50)

);

GO



-- STEP 3: Import CSV

-- UPDATE the path below to match your machine

BULK INSERT dbo.reviews

FROM 'C:\Users\kumar\Downloads\gold_master_export.csv'

WITH (

    FORMAT = 'CSV',

    FIRSTROW = 2,

    FIELDTERMINATOR = ',',

    ROWTERMINATOR = '0x0a',

    CODEPAGE = '65001',

    TABLOCK

);

GO



-- STEP 4: Verify import

SELECT

    app,

    COUNT(*) AS total_reviews,

    ROUND(
        AVG(CAST(rating AS FLOAT)),
        2
    ) AS avg_rating,

    MIN(date) AS earliest_review,

    MAX(date) AS latest_review

FROM dbo.reviews

GROUP BY app

ORDER BY app;

GO
"""


print(script)


-- ================================================
-- UPI Analysis -- SSMS Setup Script
-- Run this entire script in SSMS
-- ================================================

-- STEP 1: Create Database

USE master;
GO

IF NOT EXISTS (
    SELECT name 
    FROM sys.databases
    WHERE name = N'UPI_Analysis'
)
BEGIN
    CREATE DATABASE UPI_Analysis;
END
GO


USE UPI_Analysis;
GO


-- STEP 2: Create reviews table

DROP TABLE IF EXISTS dbo.reviews;

CREATE TABLE dbo.reviews (

    review_id    NVARCHAR(100),

    app          NVARCHAR(50),

    data_source  NVARCHAR(50),

    date         DATETIME,

    year         INT,

    year_month   NVARCHAR(10),

    rating       INT,

    review_text  NVARCHAR(MAX),

    thumbs_up    INT,

    reply_text   NVARCHAR(MAX),

    reply_date   DATETIME,

    user_name    NVARCHAR(200),

    app_version  NVARCHAR(50)

);

GO



-- STEP 3: Import CSV

-- UPDATE the path below to match your machine

BULK INSERT dbo.reviews

FROM 'C:\Users\kumar\Downloads

---
## Notebook 01 Complete

**What was built — Bronze to Silver to Gold:**

| Layer | Action | Result |
|---|---|---|
| Bronze | Loaded 7 raw files as-is | 248,000 raw rows |
| Silver | Renamed, parsed dates, removed nulls, deduplicated | 189,647 clean rows |
| Gold | Merged into one master table, validated | 189,634 unique rows |
| Export | CSV exported for SQL Server |  |
| SSMS | T-SQL script printed above | Run it to create  DB |

**Expected output after running SSMS script:**


**Once SSMS verification passes — share the output here and we will start building the 6 T-SQL query files.**
